In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import ripser
import persim

In [2]:
########## read the data ##########

processes = ["hhvv", "hhmumu", "jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG", "hzvvBG"]
event_type = [str(i)+"_200K" for i in processes]

Xsection = [4.0360050e-02, +3.3945676e-03, 1.345150e+01, 4.423600e+00, 2.325600e+00, 1.506969e+02, 1.224052e+01, 9.618000e+00, 6.473005e+00, 7.4722023e-01]     ### unit: fb
Luminosity = 1000   ### unit: fb^-1
simulation_num = 200000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Mass", "VLCjetR10N2.Constituents", "EFlowTrack.fUniqueID", "EFlowTrack.PT", "EFlowTrack.Eta", "EFlowTrack.Phi", "EFlowTrack.PID", "EFlowPhoton.fUniqueID", "EFlowPhoton.ET", "EFlowPhoton.Eta", "EFlowPhoton.Phi", "EFlowNeutralHadron.fUniqueID", "EFlowNeutralHadron.ET", "EFlowNeutralHadron.Eta", "EFlowNeutralHadron.Phi"]

##### set data path #####

event_path = []
for type in event_type:
    event_path.append("/data/mucollider/two_boosted/testdata/" + type + "/delphes_output.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

/usr/local/lib/python3.8/dist-packages/awkward/array/base.py:394: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return cls.numpy.array(value, copy=False)


Time:29.8136 (s)
Time:59.1557 (s)
Time:93.1112 (s)
Time:122.7063(s)
Time:150.4879(s)
Time:169.8641(s)
Time:190.0082(s)
Time:207.1707(s)
Time:228.6427(s)
Time:243.3731(s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]


In [4]:
def get_particle_info(event):
    ##### set jet constituents ID #####

    leadingJ_constituents = event[features.index("VLCjetR10N2.Constituents")][0][:-1]
    subleadingJ_constituents= event[features.index("VLCjetR10N2.Constituents")][1][:-1]

    ##### construct infromation for all EFlow particles in this events and sort by PT #####

    all_ID = np.concatenate((event[features.index("EFlowTrack.fUniqueID")], event[features.index("EFlowPhoton.fUniqueID")], event[features.index("EFlowNeutralHadron.fUniqueID")]))
    all_PT = np.concatenate((event[features.index("EFlowTrack.PT")], event[features.index("EFlowPhoton.ET")], event[features.index("EFlowNeutralHadron.ET")]))
    all_Eta = np.concatenate((event[features.index("EFlowTrack.Eta")], event[features.index("EFlowPhoton.Eta")], event[features.index("EFlowNeutralHadron.Eta")]))
    all_Phi = np.concatenate((event[features.index("EFlowTrack.Phi")], event[features.index("EFlowPhoton.Phi")], event[features.index("EFlowNeutralHadron.Phi")]))
    all_PID = np.concatenate((event[features.index("EFlowTrack.PID")], np.full(len(event[features.index("EFlowPhoton.fUniqueID")]), 22), np.full(len(event[features.index("EFlowNeutralHadron.fUniqueID")]), 4)))

    #####                          categorize PID                      #####
    ##### 1:electron 2:muon 3:photon 4:neutral hadron 5:charged hadron #####

    condlist = [abs(all_PID)==11, abs(all_PID)==13, abs(all_PID)==22, abs(all_PID)==4]
    choicelist = [1, 2, 3, 4]
    all_PID = np.select(condlist, choicelist, 5)

    sort_order = np.argsort(all_PT)[::-1]   ### from large PT to low PT

    all_ID = all_ID[sort_order]
    all_PT = all_PT[sort_order]
    all_Eta = all_Eta[sort_order]
    all_Phi = all_Phi[sort_order]
    all_PID = all_PID[sort_order]

    ##### find Jet particle in EFlow particle #####
    if len(leadingJ_constituents)>20: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0][0:20]
    else: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0]

    if len(subleadingJ_constituents)>20: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0][0:20]
    else: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0]

    leadingJ_PT, subleadingJ_PT = all_PT[where_leading], all_PT[where_subleading]
    leadingJ_Eta, subleadingJ_Eta = all_Eta[where_leading], all_Eta[where_subleading]
    leadingJ_Phi, subleadingJ_Phi = all_Phi[where_leading], all_Phi[where_subleading]
    leadingJ_PID, subleadingJ_PID = all_PID[where_leading], all_PID[where_subleading]

    leadingJ_info, subleadingJ_info = np.stack((leadingJ_PT, leadingJ_Eta, leadingJ_Phi, leadingJ_PID), axis=1).ravel(), np.stack((subleadingJ_PT, subleadingJ_Eta, subleadingJ_Phi, subleadingJ_PID), axis=1).ravel()

    ##### do zero padding #####

    particle_info = np.zeros(2*20*4)
    particle_info[0:len(leadingJ_info)] = leadingJ_info
    particle_info[80:80+len(subleadingJ_info)] = subleadingJ_info
    
    return particle_info

In [5]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 200000 hhvv events. Corresponding cross section: 0.04036005 (fb)
There are 200000 hhmumu events. Corresponding cross section: 0.0033945675999999996 (fb)
There are 200000 jjBG_ptcut events. Corresponding cross section: 13.4515 (fb)
There are 200000 ttBG_ptcut events. Corresponding cross section: 4.4236 (fb)
There are 200000 bbBG_ptcut events. Corresponding cross section: 2.3256 (fb)
There are 200000 wwBG events. Corresponding cross section: 150.6969 (fb)
There are 200000 wwvvBG events. Corresponding cross section: 12.24052 (fb)
There are 200000 zzBG events. Corresponding cross section: 9.618 (fb)
There are 200000 zzvvBG events. Corresponding cross section: 6.473005 (fb)
There are 200000 hzvvBG events. Corresponding cross section: 0.74722023 (fb)

After 2 fat jet selection:
There are 200000 hhvv events. Corresponding cross section: 0.04036005 (fb)
There are 200000 hhmumu events. Corresponding cross section: 0.0033945675999999996 (fb)
There are 200000 jjBG_

In [6]:
##### generate particle information #####

particle_info = []

for process in processes:
    for event in events[processes.index(process)]:
        particle_info.append(get_particle_info(event))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:293.7563(s)
Time:305.7987(s)
Time:308.8578(s)
Time:312.1684(s)
Time:315.6903(s)
Time:319.9460(s)
Time:328.1799(s)
Time:333.2158(s)
Time:343.1571(s)
Time:354.0485(s)


In [7]:
particle_info[0]

array([57.7942009 ,  0.25657213, -1.22692025,  5.        , 55.22911453,
        0.26472241, -1.2725848 ,  4.        , 45.6352005 ,  0.26360634,
       -1.24177134,  3.        , 37.06074524,  0.279773  , -1.26355922,
        5.        , 29.84007835,  0.28971592, -1.24037182,  5.        ,
       19.4640789 ,  0.24036421, -1.21871042,  5.        , 18.76408768,
        0.26847839, -1.23071086,  4.        , 18.48882294,  0.24376814,
       -1.2526412 ,  3.        , 18.47137451,  0.28913304, -1.2592814 ,
        3.        , 15.6185894 ,  0.26435408, -1.25017345,  5.        ,
       15.1805191 ,  0.49698836, -1.24182522,  5.        , 15.00228691,
        0.26513681, -1.25248444,  3.        , 14.62330532,  0.26292637,
       -1.26760995,  5.        , 12.86481476,  0.21301961, -1.30285776,
        5.        , 12.22763348,  0.57616228, -0.69187367,  5.        ,
       11.85291672,  0.26555347, -1.26936042,  3.        , 11.17086601,
        0.18152386, -1.25278187,  4.        , 10.16321087,  0.16

In [8]:
##### output test data #####

d = {'particle_info': particle_info}
df = pd.DataFrame(data=d)
df

,particle_info
0,"[57.7942008972168, 0.2565721273422241, -1.2269..."
1,"[38.749298095703125, -0.3129907548427582, -1.6..."
2,"[92.46818542480469, -0.3280430734157562, -0.98..."
3,"[35.78585433959961, -0.6381620168685913, 2.238..."
4,"[53.761505126953125, 0.3000965118408203, -1.54..."
...,...
347533,"[73.53440856933594, -0.2364083230495453, 0.097..."
347534,"[61.212764739990234, -1.7068015336990356, -1.9..."
347535,"[108.36914825439453, -0.5344061255455017, 1.06..."
347536,"[57.315345764160156, 0.8443825840950012, -3.05..."


In [9]:
import h5py
h5f = h5py.File('/data/mucollider/two_boosted/test_LLfeature.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")
h5f.close()

已寫入 Dataset: particle_info
